# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsiddan25/program/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*



I chose the Refresh/Content opportunity scoring because it answers a very important question of which pages should be reviewed for refresh. I felt this has more practical relevance as identifying which pages are susceptible to declining and therefore needs attention offers a working solution to head in the direction of fixing the declining traffic and push it back into an active state. The work did in notebooks 1 and 2 builds on this idea, so I want to explore this lane and go beyound.


In [3]:
# This cell is for CODE (numbers, a query, a check)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

It improves the decision of which pages need to be refreshed based on the likeliness of decline. If a page has lower traffic, certain features such as low CTR, poor position, etc will be identified that will be attributed to declining pages and thus a score for refresh will be given. The higher this score is, it means it is more likely to decline, thus it needs to have higher priority to refresh the page. A human reviewer acts on this decision. A wrong recommendation wastes the time and unnecesarily leads to attention on a page which does not require attention. Thus, pages which do require attention will not be refreshed, thus delaying the time it will be refreshed.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

My lane focuses on Refresh opportunities. Based on notebook 2, it was found that out of 30,000 pages, the decline rate is 0. 542. The hand rule found that for Precision@20 it was 0.9 and for Precision@50 it was  0.68. So, it can be seen that as the number of pages increases, the precision of the number of pages which actually decline decreases. Thus, this allows opporunity for a model to be used, in which we can analze what score the model gives and compare that to the hand rule.




In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run Al

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))


stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]


def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

    from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

# Splitting the pages into training set and testing set
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Split the original dataframe the same way
_, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=y
)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)

print(export_text(tree, feature_names=features))


tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df_test["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")


30000 pages |  declining rate: 0.542
Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.65
|   |   |--- class: 0
|   |--- avg_position >  0.65
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 344.50
|   |   |--- class: 1
|   |--- content_age_days >  344.50
|   |   |--- class: 0

Precision@20:  hand rule 0.450   vs   tree 0.500
Precision@50:  hand rule 0.580   vs   tree 0.540


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

The work will be able to help support a decision on which pages require attention/need to be refreshed that can help a human reviewer. But, it does not guarantee that refreshing the pages the model suggests will definitely lead to increased traffic. It is merely a suggestion by the model as the model observes the data it was provided with and does its best to give a recommendation. But the recommendation itself does not give a guarantee.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.